In [7]:
!pip3 install openai chromadb -q


In [36]:
import os
import json
from openai import OpenAI
import chromadb

from google.colab import userdata
api =userdata.get('Groq_apk')

client = OpenAI(
    api_key= api,
    base_url="https://api.groq.com/openai/v1"
)
print("client created successfully")


client created successfully


In [37]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "What is Nova Tech Solutions company work policy regarding work from home?"
        }
    ]
)

print(response.choices[0].message.content)

Below is a **sample Work‑From‑Home (WFH) Policy** that a company called **Nova Tech Solutions** could adopt.  
Feel free to tailor it to your own organization’s culture, legal requirements, and operational needs.  

---

## Nova Tech Solutions – Work‑From‑Home (WFH) Policy

| Section | Description |
|--------|-------------|
| **Purpose** | To provide a flexible, consistent, and productive remote‑work environment that protects both employees and the company. |
| **Scope** | Applies to all full‑time, part‑time, and contract employees who are approved for WFH. Excludes temporary or contract workers whose terms do not allow remote work. |
| **Policy Statement** | Nova Tech Solutions supports a hybrid work model: employees may work from home **up to 2 days per week** (subject to manager approval) while remaining onsite for key team events, training, or when required by job function. |

---

### 1. Eligibility & Approval

| Criteria | Details |
|----------|---------|
| **Job Function** | Rol

# **load and chunk the document**

In [21]:
with open("/content/company_hr_policy.txt", "r" ) as f:
  hr_document = f.read()

with open("/content/engineering_standards.txt", "r" ) as f:
  engineering_standards = f.read()

with open("/content/onboarding_guide.txt", "r" ) as f:
  onboarding_guide = f.read()

with open("/content/product_knowledge_base.txt", "r" ) as f:
  product_knowledge_base = f.read()

with open("/content/security_policy.txt", "r" ) as f:
  security_policy = f.read()


In [22]:
def chunk_documents ( text, source_name):
  paragraphs = text.strip().split("\n\n")
  chunks =[]
  for para in paragraphs:
    para = para.strip()
    if len(para)< 50:
      continue
    if para.startswith("====="):
      continue

    chunks.append({
        "text":para,
        "source": source_name
    })
  return chunks

hr_chunks = chunk_documents(hr_document,"HR policy")
engineering_standards_chunks = chunk_documents(engineering_standards,"eg")
onboarding_guide_chunks = chunk_documents(onboarding_guide,"on boarding policy")
product_knowledge_base_chunks = chunk_documents(product_knowledge_base,"product")
security_policy_chunks = chunk_documents(security_policy,"security")


In [23]:
all_chunks = hr_chunks + engineering_standards_chunks +onboarding_guide_chunks+product_knowledge_base_chunks+security_policy_chunks
print(all_chunks)
print(len(all_chunks))

[{'text': 'NovaTech Solutions — Employee Handbook & HR Policy\nVersion 3.2 | Last Updated: January 2026', 'source': 'HR policy'}, {'text': 'Annual Leave:\nAll full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave accrues at the rate of 2 days per month. New employees can start using accrued leave after completing 3 months of service. Unused leave up to 10 days can be carried forward to the next year. Any leave beyond 10 days will lapse on December 31st.', 'source': 'HR policy'}, {'text': 'Sick Leave:\nEmployees are entitled to 12 days of sick leave per year. Sick leave for more than 3 consecutive days requires a medical certificate from a registered medical practitioner. Sick leave cannot be carried forward or encashed. In case of extended illness beyond 12 days, employees may apply for medical leave without pay, subject to HR approval.', 'source': 'HR policy'}, {'text': 'Casual Leave:\nEmployees are entitled to 6 days of casual leave per year. Casua

#Storing the chunks in chromadb


In [27]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="company_docs1")

In [28]:
documents =[]
ids =[]
metadata =[]

for i, chunk in enumerate(all_chunks):
  documents.append(chunk['text'])
  ids.append(f"chunks_{i}")
  metadata.append({"source":chunk["source"]})

collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadata
)

# Retriving the data

In [29]:
def retrieve(question , n_results ):
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )

  return results["documents"][0],results['metadatas'][0]

chunks , sources = retrieve("whats is the work from home policy?" , 5)
for i in range (len(chunks)):
  print(f"------Chunks{i+1}----------")
  print(f"Sources: {sources[i]['source']}")
  print(f"text: {chunks[i]}")
  print()


------Chunks1----------
Sources: HR policy
text: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.

------Chunks2----------
Sources: HR policy
text: NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: January 2026

------Chunks3----------
Sources: security
text: Internet Usage:
- Company internet is primarily for work purposes
- Limited personal use is acceptable during breaks
- The following are strictly prohibited: downloading copyrighted content (movies, software), accessing explicit content, cryptocurrency mining, running personal businesses on company infrastructure

------Chunks4----------
Sources: HR policy
text: Regular WFH:
Employees may work from home up to 2 days per week. The preferred WFH days are Wednesday and Friday, though teams may adjust based on 

Ask rag

In [32]:
def ask_rag(question , n_results , verbose=True):
  chunks , sources = retrieve(question , n_results)
  if verbose:
    print(f"\n{'═' * 60}")
    print(f"❓ Question: {question}")
    print(f"{'─' * 60}")
    print(f"📄 Retrieved {len(chunks)} chunks:")
    for i, (chunk, source) in enumerate(zip(chunks, sources)):
        print(f"   [{source['source']}] {chunk[:80]}...")
    print(f"{'─' * 60}")


  context ="\n\n".join(chunks)

  messages =[
      {"role": "system",
       "content": (
           "You are a helpful assistant that answers questions based ONLY on the provided context."
           "If the context does not contain enough informaiton to answer the question"
           "say I Don't have enough informaiton to answer this question"
           "Do not make up any information and always be conscise."
           )},
      {
          "role": "user",
          "content": f"Context: \n{context}\n\nQuestion: {question}"
      }
  ]

  from google.colab import userdata

  groq_api = userdata.get('Groq_apk')
  groq_client = OpenAI(api_key = groq_api,
                       base_url ="https://api.groq.com/openai/v1")

  response = groq_client.chat.completions.create(
      model="openai/gpt-oss-20b",
      messages = messages,
      temperature =0.2

  )


  answer = response.choices[0].message.content

  if verbose:
    print(f"Answer: {answer}")
    print(f"{'='*60}")

  return answer


print("Rag Piepline is ready!!")



Rag Piepline is ready!!


# test rag pipeline

In [35]:
ask_rag("what is the work from home policy?" ,3)


════════════════════════════════════════════════════════════
❓ Question: what is the work from home policy?
────────────────────────────────────────────────────────────
📄 Retrieved 3 chunks:
   [HR policy] Eligibility:
All employees who have completed their probation period (6 months) ...
   [HR policy] NovaTech Solutions — Employee Handbook & HR Policy
Version 3.2 | Last Updated: J...
   [HR policy] Regular WFH:
Employees may work from home up to 2 days per week. The preferred W...
────────────────────────────────────────────────────────────
Answer: **Work‑From‑Home (WFH) Policy – NovaTech Solutions (Version 3.2, Jan 2026)**  

- **Eligibility**  
  - All employees who have completed the 6‑month probation period are eligible for WFH.  
  - Employees still in probation may request WFH only in exceptional circumstances, with approval from their manager and HR.

- **Regular WFH**  
  - Eligible employees may work from home up to **2 days per week**.  
  - Preferred WFH days are **Wednes

'**Work‑From‑Home (WFH) Policy – NovaTech Solutions (Version\u202f3.2, Jan\u202f2026)**  \n\n- **Eligibility**  \n  - All employees who have completed the 6‑month probation period are eligible for WFH.  \n  - Employees still in probation may request WFH only in exceptional circumstances, with approval from their manager and HR.\n\n- **Regular WFH**  \n  - Eligible employees may work from home up to **2 days per week**.  \n  - Preferred WFH days are **Wednesday and Friday**, though teams can adjust based on project needs.  \n  - Employees must be available during **core working hours (10:00\u202fAM – 6:00\u202fPM IST)** on WFH days.'